# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library. All dataset entities, such as record sets, fields, and columns, are referenced via their `@id` identifiers per the Croissant specification and best practices.

### Dataset Source

The dataset Croissant schema is available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

In this section, we'll load the dataset metadata and available records using `mlcroissant`. This will allow us to inspect the overall structure, available record sets, fields, and data needed for downstream analysis.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL (Croissant JSON-LD schema)
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (avoid subscripting; use attributes or .to_json())
md = dataset.metadata
md_json = md.to_json()

print(f"Dataset Name: {md_json.get('name')}")
print(f"Description: {md_json.get('description')}")

## 2. Data Overview

Let's review what record sets, fields and columns are available. This information is critical for selecting and extracting the parts of the data we're interested in.

`mlcroissant` allows exploration of record sets and their fields via the dataset metadata structure. We'll collect the `@id` of each record set, and for each, summarize its available fields and columns.

In [ ]:
# Helper: pretty-printing
pp = pprint.PrettyPrinter(indent=2)

# Get all record set entities
record_sets = []

if hasattr(md, 'record_sets') and md.record_sets:
    record_sets = md.record_sets
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        print(f"- Record Set @id: {rs.id}")
        print(f"  Name: {getattr(rs, 'name', None)}")
        print(f"  Description: {getattr(rs, 'description', None)}")
        # Display fields by @id:
        print(f"  Fields:")
        if hasattr(rs, 'fields') and rs.fields:
            for f in rs.fields:
                print(f"    - Field @id: {getattr(f, 'id', None)}, Name: {getattr(f, 'name', None)}, Description: {getattr(f, 'description', None)}")
        # Display columns by @id:
        if hasattr(rs, 'columns') and rs.columns:
            print(f"  Columns:")
            for c in rs.columns:
                print(f"    - Column @id: {getattr(c, 'id', None)}, Name: {getattr(c, 'name', None)}, dataType: {getattr(c, 'data_type', None)}")
        print()
else:
    print("No record sets found in the dataset metadata.")

## 3. Data Extraction

Load data for each record set identified in the previous step. We'll use the `@id` for each record set to extract its records using `dataset.records(record_set=...)`. Each field and column is also referenced by its unique `@id`.

Data from each record set is loaded into a pandas DataFrame for analysis.

In [ ]:
# Build a list of record set @ids
# Will be auto-discovered from metadata if present

record_set_ids = []
if hasattr(md, 'record_sets') and md.record_sets:
    record_set_ids = [rs.id for rs in md.record_sets]
else:
    print("No record sets found.")

# Dictionary to hold DataFrames for each record set
dfs = {}

for rs_id in record_set_ids:
    print(f"Loading records for Record Set @id: {rs_id}")
    # records() API expects record_set to be @id
    recs = list(dataset.records(record_set=rs_id))
    if recs:
        dfs[rs_id] = pd.DataFrame(recs)
        print(f"  Loaded {len(recs)} records.")
        print(f"  Fields: {list(dfs[rs_id].columns)}\n")
    else:
        print("  No records found.")

# For demonstration, pick the first available record set (if any)
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Columns/Fields for Record Set @id: {main_rs_id}")
    print(dfs[main_rs_id].columns.tolist() if main_rs_id in dfs else 'No data loaded.')
    if main_rs_id in dfs:
        dfs[main_rs_id].head()
else:
    print("Cannot proceed, no record sets with data.")

## 4. Exploratory Data Analysis (EDA)

Let’s perform basic EDA on a chosen numeric field. We'll select a numeric field from one of the DataFrames by its `@id`, filter rows, normalize the field, and group by a chosen attribute (all by `@id`).

> **Note:** As this dataset may include missing data and sensitive fields (see metadata), always respect privacy constraints when analyzing outputs.

We'll demonstrate:
- Filtering by a numeric field's value,
- Normalizing that field,
- Grouping by a selected group field (if present).

In [ ]:
# Choose the main DataFrame and fields based on available data
if record_set_ids and record_set_ids[0] in dfs:
    rs_id = record_set_ids[0]
    df = dfs[rs_id]
    print(f"Operating on Record Set @id: {rs_id}")
    print(f"Available columns (by @id): {list(df.columns)}")

    # Attempt to automatically select a numeric field (@id) to demo (fall back if not present)
    numeric_field_id = None
    group_field_id = None
    # Simple auto-selection: check first numeric column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Attempt to select group by a field with 'ward' or 'region' or 'gender' (if present)
    for col in df.columns:
        col_lower = col.lower() if isinstance(col, str) else str(col).lower()
        if any(k in col_lower for k in ['ward', 'region', 'county', 'gender']):
            group_field_id = col
            break

    if numeric_field_id:
        print(f"Using numeric field (by @id): {numeric_field_id}")
        threshold = df[numeric_field_id].mean() # dynamic threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} records.")
        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, norm_col]].head())
        # Group by selected field
        if group_field_id and group_field_id in filtered_df.columns:
            print(f"\nGrouping by field (by @id): {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable group field found in columns.")
    else:
        print("No numeric field detected for analysis.")
else:
    print("No extracted records available for EDA.")

## 5. Visualization

Let’s generate basic visualizations to understand numeric field distributions and group differences (where available). All fields are referenced using their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use the filtered_df and columns selected in previous EDA
if 'filtered_df' in locals() and not filtered_df.empty and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=20)
    plt.title(f'Distribution of Numeric Field (@id: {numeric_field_id})')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    # If grouped_df exists, plot group means
    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, ci=None)
        plt.title(f'Mean of Numeric Field ({numeric_field_id}) by Group (@id: {group_field_id})')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Visualization not possible: filtered_df or required field(s) missing.")

## 6. Conclusion

In this notebook, we loaded, explored, and visualized the FAIR^2 dataset using the `mlcroissant` library and referenced all data entities by their `@id` fields for reproducibility. We:
- Inspected dataset structure and metadata,
- Listed available record sets, fields, and their IDs,
- Extracted data into DataFrames,
- Performed filtering, normalization, and group-wise analysis of numeric fields,
- Created visualizations to interpret key distributions.

This approach can be extended for richer analysis, predictive modeling, or policy insights on knowledge adoption in rangeland management, while always referencing fields and groups by their canonical `@id` from the Croissant schema.

For further analysis or to process additional record sets, repeat the above steps using the desired `@id` values as demonstrated.